# Colposcopy Lesion Segmentation — End-to-End Pipeline

Run every cell top to bottom, in order, on a fresh runtime (**Runtime -> Restart session and run all** if you're re-running after an earlier failed attempt — mixing a restarted runtime with leftover state from a previous one is exactly what caused the working-directory issues earlier).

**Before running:** Runtime -> Change runtime type -> T4 GPU (or better).


## 1. Setup: mount Drive (optional but recommended) and set the project root

In [ ]:
# Set to True to persist data/checkpoints/outputs across Colab sessions.
# Strongly recommended -- otherwise everything is wiped when the runtime recycles.
USE_DRIVE = True

import os

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/colposcopy'
else:
    PROJECT_ROOT = '/content/colposcopy'

os.makedirs(PROJECT_ROOT, exist_ok=True)
os.chdir(PROJECT_ROOT)
print('Working directory set to:', os.getcwd())


## 2. Get the code
Upload `colposcopy_seg_pipeline.zip` using the file browser on the left (into `/content`, not into `PROJECT_ROOT`), then run this cell. It's safe to re-run -- `unzip -o` overwrites, it won't duplicate anything.

In [ ]:
ZIP_PATH = '/content/colposcopy_seg_pipeline.zip'

if not os.path.exists(ZIP_PATH):
    try:
        from google.colab import files
        print('Zip not found at', ZIP_PATH, '-- pick it from your computer:')
        uploaded = files.upload()
        ZIP_PATH = '/content/' + next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(f'Upload the zip to {ZIP_PATH} first.')

!unzip -o "{ZIP_PATH}" -d "{PROJECT_ROOT}"
!ls "{PROJECT_ROOT}"


## 3. Get the data
Clones straight into `PROJECT_ROOT/annocerv_raw` (skips if already present, e.g. from a previous session on the same Drive). The sanity-check `ls` confirms the path `prepare_data.py` needs actually exists **before** we rely on it -- this is exactly the check that would have caught the earlier working-directory mismatch immediately instead of two cells later.

In [ ]:
RAW_DIR = os.path.join(PROJECT_ROOT, 'annocerv_raw')

if not os.path.exists(os.path.join(RAW_DIR, 'dataset', 'swede_scores.csv')):
    !git clone --depth=1 https://github.com/iclx/AnnoCerv "{RAW_DIR}"
else:
    print('annocerv_raw already present, skipping clone.')

# sanity check -- must print a real file, not an error, before continuing
!ls -la "{RAW_DIR}/dataset/swede_scores.csv"


## 4. Install dependencies

In [ ]:
%cd {PROJECT_ROOT}
!pip install -q -r requirements.txt


## 5. Prepare data (unify + parse masks + case-level split)
Fast, CPU-only, a couple minutes. Re-run any time you change `--swede_threshold` or crop settings -- it's fully deterministic and safe to overwrite.

In [ ]:
!python src/prepare_data.py --raw_dir "{RAW_DIR}/dataset" --out_dir data --swede_threshold 5


## 6. Train
Uses GPU automatically if available (check the first printed line says `Device: cuda`, not `Device: cpu` -- if it says cpu, fix the runtime type in Runtime -> Change runtime type before continuing). Checkpoints save to `outputs/best_model.pt` every time val_dice improves, so an interrupted run doesn't lose progress -- you can lower `training.epochs` in `configs/config.yaml` and re-run this cell to continue experimenting without waiting for the full 100 every time.

In [ ]:
!python src/train.py --config configs/config.yaml


## 7. Evaluate on the held-out test set

In [ ]:
!python src/evaluate.py --config configs/config.yaml


## 8. Inspect results: training curves + real prediction overlays
This is the important one -- don't stop at the Dice number above. Generates `outputs/training_curves.png` and `outputs/qualitative_predictions_test.png`.

In [ ]:
!python src/inspect_results.py --config configs/config.yaml --n_samples 8


In [ ]:
import os
assert os.path.exists('outputs/training_curves.png'), (
    'training_curves.png missing -- the cell above must have errored; '
    'scroll up and check its output before continuing.'
)
from IPython.display import Image, display
display(Image('outputs/training_curves.png'))


In [ ]:
display(Image('outputs/qualitative_predictions_test.png'))


## 9. Full numeric results
`dice_by_grade` is the number worth checking most closely -- confirms whether high-grade lesions (the clinically important ones) are being segmented at least as well as low-grade, not worse.

In [ ]:
import json
results = json.load(open('outputs/test_results.json'))
print(json.dumps(results, indent=2))


In [ ]:
import pandas as pd
history = pd.read_csv('outputs/training_history.csv')
pd.set_option('display.max_rows', None)
history


## 10. (Optional) Inspect train/val predictions too
Same qualitative grid, but on splits the model *has* seen -- useful for spotting overfitting: if train predictions look great and val/test don't, that's the gap to close (more data, more augmentation, more dropout), not a threshold-tuning problem.

In [ ]:
!python src/inspect_results.py --config configs/config.yaml --n_samples 6 --split train
display(Image('outputs/qualitative_predictions_train.png'))
